In [1]:
%pip install pyspark duckdb


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Configuração da sessão do spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf
import time

Config = SparkConf()
Config.set("spark.sql.repl.eagerEval.enabled", True)
Config.set("spark.sql.repl.eagerEval.maxNumRows", "20")
Config.set("spark.sql.repl.eagerEval.truncate", "-1")
Config.set("spark.driver.memory","5G")
Config.set("spark.memory.fraction", 0.9)


Config.set("spark.sql.shuffle.partitions", 100)
Config.set("spark.default.parallelism", 200)

# Iniciando a sessão do Spark (O "cérebro" da aplicação)
spark = SparkSession.builder.config(conf=Config).master("local[*]").appName("OverviewSpark").getOrCreate()

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 55566)
Traceback (most recent call last):
  File "/Users/fabiokishino/.pyenv/versions/3.12.7/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/fabiokishino/.pyenv/versions/3.12.7/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/Users/fabiokishino/.pyenv/versions/3.12.7/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/fabiokishino/.pyenv/versions/3.12.7/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/Users/fabiokishino/Documents/Dev/pos-data-science/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/Users/fabiokishino/Documents/Dev/pos-data-scienc

In [6]:
spark.sparkContext.getConf().get("spark.driver.memory")

'5G'

In [ ]:
spark.stop()

Uma apresentação de todos os parâmetros default que estão configurados

In [7]:
# 1. Criando um DataFrame simples
data = [("Voo 101", "GOL", 500.0), ("Voo 202", "LATAM", 700.0),
        ("Voo 303", "AZUL", 450.0), ("Voo 404", "GOL", 550.0)]
columns = ["id_voo", "companhia", "preco"]

df = spark.createDataFrame(data, columns)

print("--- PASSO 1: Definindo as Transformações (Lazy) ---")

# TRANSFORMAÇÃO 1: Filtrar apenas voos da GOL
# O Spark não lê os dados aqui, ele apenas anota o filtro no plano de execução.
df_filtrado = df.filter(col("companhia") == "GOL")

# TRANSFORMAÇÃO 2: Criar uma coluna com o nome da empresa em maiúsculo
df_final = df_filtrado.withColumn("companhia_upper", upper(col("companhia")))

print("Transformações anotadas no plano de execução (DAG).")
print("Observe que nada foi processado ainda.")

print("\n--- PASSO 2: Executando uma AÇÃO ---")
print("Agora, ao chamar o .show(), o Spark executará todas as etapas acima de uma vez.")

# AÇÃO: .show() dispara a execução real
start_time = time.time()
df_final.show()
print(f"Execução concluída em: {time.time() - start_time:.4f} segundos")

--- PASSO 1: Definindo as Transformações (Lazy) ---
Transformações anotadas no plano de execução (DAG).
Observe que nada foi processado ainda.

--- PASSO 2: Executando uma AÇÃO ---
Agora, ao chamar o .show(), o Spark executará todas as etapas acima de uma vez.


+-------+---------+-----+---------------+
| id_voo|companhia|preco|companhia_upper|
+-------+---------+-----+---------------+
|Voo 101|      GOL|500.0|            GOL|
|Voo 404|      GOL|550.0|            GOL|
+-------+---------+-----+---------------+

Execução concluída em: 4.5405 segundos


In [8]:
df_final

id_voo,companhia,preco,companhia_upper
Voo 101,GOL,500.0,GOL
Voo 404,GOL,550.0,GOL


In [9]:
# Registrando o DataFrame como uma tabela temporária (View)
df.createOrReplaceTempView("voos_table")

# Executando uma consulta SQL diretamente
print("\n--- PASSO 3: Usando Spark SQL ---")
resultado_sql = spark.sql("""
    SELECT companhia, AVG(preco) as preco_medio
    FROM voos_table
    GROUP BY companhia
""")

# Note que o Spark SQL também é LAZY. A execução só ocorre no .show()
resultado_sql


--- PASSO 3: Usando Spark SQL ---


companhia,preco_medio
GOL,525.0
LATAM,700.0
AZUL,450.0


In [10]:
df.groupBy("companhia").agg(
    avg("preco").alias("preco_medio")
)


companhia,preco_medio
GOL,525.0
LATAM,700.0
AZUL,450.0


# Testando dados Grandes de verdade

https://dadosabertos.tse.jus.br/dataset/resultados-2022-boletim-de-urna

In [11]:
!wget https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2022/buweb/bweb_1t_RJ_051020221321.zip

--2026-05-24 19:00:19--  https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2022/buweb/bweb_1t_RJ_051020221321.zip
Resolving cdn.tse.jus.br (cdn.tse.jus.br)... 187.32.204.94, 8.242.50.94
Connecting to cdn.tse.jus.br (cdn.tse.jus.br)|187.32.204.94|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 148585987 (142M) [application/zip]
Saving to: ‘bweb_1t_RJ_051020221321.zip’

bweb_1t_RJ_05102022 100%[===================>] 141.70M  2.76MB/s    in 52s     

2026-05-24 19:01:11 (2.74 MB/s) - ‘bweb_1t_RJ_051020221321.zip’ saved [148585987/148585987]



In [12]:
!unzip -o bweb_1t_RJ_051020221321.zip

Archive:  bweb_1t_RJ_051020221321.zip
  inflating: bweb_1t_RJ_051020221321.csv  
  inflating: leiame-boletimurnaweb.pdf  


In [13]:
df_bweb = spark.read.csv(
    'bweb_1t_RJ_051020221321.csv',
    header=True,
    inferSchema=True,
    sep=";"
)
df_bweb

26/05/24 19:03:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_LOCAL_VOTACAO,CD_CARGO_PERGUNTA,DS_CARGO_PERGUNTA,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,DT_BU_RECEBIDO,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCOES,CD_TIPO_URNA,DS_TIPO_URNA,CD_TIPO_VOTAVEL,DS_TIPO_VOTAVEL,NR_VOTAVEL,NM_VOTAVEL,QT_VOTOS,NR_URNA_EFETIVADA,CD_CARGA_1_URNA_EFETIVADA,CD_CARGA_2_URNA_EFETIVADA,CD_FLASHCARD_URNA_EFETIVADA,DT_CARGA_URNA_EFETIVADA,DS_CARGO_PERGUNTA_SECAO,DS_AGREGADAS,DT_ABERTURA,DT_ENCERRAMENTO,QT_ELEITORES_BIOMETRIA_NH,DT_EMISSAO_BU,NR_JUNTA_APURADORA,NR_TURMA_APURADORA
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,544,Elei��o Geral Federal 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,1,Presidente,13,PT,Partido dos Trabalhadores,02/10/2022 19:28:41,341,250,91,1,APURADA,1,Nominal,13,LULA,137,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,1 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,544,Elei��o Geral Federal 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,1,Presidente,22,PL,Partido Liberal,02/10/2022 19:28:41,341,250,91,1,APURADA,1,Nominal,22,JAIR BOLSONARO,70,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,1 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,544,Elei��o Geral Federal 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,1,Presidente,15,MDB,Movimento Democr�tico Brasileiro,02/10/2022 19:28:41,341,250,91,1,APURADA,1,Nominal,15,SIMONE TEBET,18,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,1 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,544,Elei��o Geral Federal 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,1,Presidente,12,PDT,Partido Democr�tico Trabalhista,02/10/2022 19:28:41,341,250,91,1,APURADA,1,Nominal,12,CIRO GOMES,12,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,1 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,544,Elei��o Geral Federal 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,1,Presidente,-1,#NULO#,#NULO#,02/10/2022 19:28:41,341,250,91,1,APURADA,2,Branco,95,Branco,2,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,1 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,544,Elei��o Geral Federal 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,1,Presidente,-1,#NULO#,#NULO#,02/10/2022 19:28:41,341,250,91,1,APURADA,3,Nulo,96,Nulo,4,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,1 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,544,Elei��o Geral Federal 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,1,Presidente,30,NOVO,Partido Novo,02/10/2022 19:28:41,341,250,91,1,APURADA,1,Nominal,30,FELIPE D'AVILA,7,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,1 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,546,Elei��es Gerais Estaduais 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,3,Governador,-1,#NULO#,#NULO#,02/10/2022 19:28:41,341,250,91,1,APURADA,3,Nulo,96,Nulo,11,2208140,160.260.237.153.996.625.,558.525,E356201E,19/09/2022 15:19:00,3 - 10,#NULO#,02/10/2022 08:00:01,02/10/2022 17:03:19,13,02/10/2022 17:05:41,-1,-1
05/10/2022,2026-05-24 15:08:07,2022,0,Elei��o Ordin�ria,406,02/10/2022,1,546,Elei��es Gerais Estaduais 2022,RJ,60011,RIO DE JANEIRO,4,10,1180,3,Governador,21,PCB,Partido Comu

In [14]:
df_bweb.count()

7107294

In [15]:
df_bweb.write.parquet("bweb/")

In [16]:
df_bweb.repartition(1).write.parquet("bweb_repart/")

In [ ]:
!wget https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2022/buweb/bweb_1t_SP_051020221321.zip

In [ ]:
!unzip -o bweb_1t_SP_051020221321.zip


In [ ]:
df_bweb_sp = spark.read.csv(
    'bweb_1t_SP_051020221321.csv',
    header=True,
    inferSchema=True,
    sep=";"
)
#df_bweb_sp.printSchema()

In [ ]:
#df_bweb_sp.repartition(1).write.parquet("bweb_sp_repart/", mode='overwrite')
df_bweb_sp.write.parquet("bweb_sp/", mode='overwrite')

In [ ]:
df_bweb_parquet = spark.read.parquet("bweb_sp/")

In [ ]:
df_bweb_parquet.printSchema()

root
 |-- DT_GERACAO: string (nullable = true)
 |-- HH_GERACAO: timestamp (nullable = true)
 |-- ANO_ELEICAO: integer (nullable = true)
 |-- CD_TIPO_ELEICAO: integer (nullable = true)
 |-- NM_TIPO_ELEICAO: string (nullable = true)
 |-- CD_PLEITO: integer (nullable = true)
 |-- DT_PLEITO: string (nullable = true)
 |-- NR_TURNO: integer (nullable = true)
 |-- CD_ELEICAO: integer (nullable = true)
 |-- DS_ELEICAO: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CD_MUNICIPIO: integer (nullable = true)
 |-- NM_MUNICIPIO: string (nullable = true)
 |-- NR_ZONA: integer (nullable = true)
 |-- NR_SECAO: integer (nullable = true)
 |-- NR_LOCAL_VOTACAO: integer (nullable = true)
 |-- CD_CARGO_PERGUNTA: integer (nullable = true)
 |-- DS_CARGO_PERGUNTA: string (nullable = true)
 |-- NR_PARTIDO: integer (nullable = true)
 |-- SG_PARTIDO: string (nullable = true)
 |-- NM_PARTIDO: string (nullable = true)
 |-- DT_BU_RECEBIDO: string (nullable = true)
 |-- QT_APTOS: integer (nullable

In [ ]:
df_bweb_parquet.write.parquet("bweb_sp_2/", mode='overwrite')

In [ ]:
df_bweb_parquet.repartition(10).write.parquet("bweb_sp_2/", mode='overwrite')

In [17]:
# Assuming 'spark' is your SparkSession object
sc = spark.sparkContext
print(sc.uiWebUrl)

http://192.168.0.27:4040


Instalando o ngrok para ter acesso ao Spark UI

In [ ]:
import os
from google.colab import userdata
token = userdata.get('NGROK_TOKEN')
os.system(f"ngrok config add-authtoken {token}")

0

In [ ]:
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc   | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null   && echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main"   | sudo tee /etc/apt/sources.list.d/ngrok.list   && sudo apt update   && sudo apt install ngrok

deb https://ngrok-agent.s3.amazonaws.com bookworm main
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:11 https://ngrok-agent.s3.amazonaws.com bookworm InRelease [20.3 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [86.4 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-